# Streamlit Demo

This notebook checks and launches the local sign-in page using the saved models.

Choose an enrolled account and take a photo. After the face passes, upload that account's voice recording, then its fingerprint image from `datasets/demo`. Each step verifies against the same account. The prepared samples represent virtual identities.


In [1]:
import sys
import json
import subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display, Code
from config import ROOT, LIVE, MODELS
from verification import load_system
assert (ROOT / 'app.py').exists(), 'The shared Streamlit application is missing.'
system = load_system()
live = pd.read_csv(LIVE / 'manifest.csv')
display(live[['display_name', 'user_id', 'original_speaker_id', 'original_finger_id', 'source_split']])
print('Saved models:', system['selected'])
print('Python:', sys.executable)

,display_name,user_id,original_speaker_id,original_finger_id,source_split
0,Abdelrahman,user_01,834f03fe,FVC2004_DB1_B:101,test
1,kareem,user_02,8e05039f,FVC2004_DB1_B:102,test
2,renad,user_03,3bfd30e6,FVC2004_DB1_B:103,test


Saved models: {'face': 'svm', 'voice': 'svm', 'fingerprint': 'svm', 'command': 'knn', 'face_voice': 'svm', 'fusion3': 'svm'}
Python: <project directory>\.venv\Scripts\python.exe


## The Actual Application Code

The code below is read directly from `app.py`. The camera supplies the face image. Each successful check reveals the next input, and the page confirms sign-in after all three checks.


In [2]:
app_source = (ROOT / 'app.py').read_text(encoding='utf-8')
compile(app_source, str(ROOT / 'app.py'), 'exec')
display(Code(app_source, language='python'))

"""Camera-first sign-in using the saved biometric verifiers."""

import streamlit as st

from config import MODELS
from verification import load_system, verify_face, verify_fingerprint, verify_voice


@st.cache_resource
def cached_system(index_modified):
    return load_system()


def check_sample(stage, verifier, sample, claim, system, attempt):
    """Reuse a decision only for the same upload and preceding steps."""
    key = f"{stage}_check"
    input_key = (attempt, sample.file_id)
    previous = st.session_state.get(key)
    if previous is None or previous[0] != input_key:
        with st.spinner(f"Checking {stage}…"):
            decision = verifier(sample.getvalue(), claim, system)
        st.session_state[key] = (input_key, decision)
    return st.session_state[key][1]


st.set_page_config(page_title="Biometric Sign-in", layout="centered")
st.title("Biometric Sign-in")

try:
    model_revision = (MODELS / "model_index.json").stat().st_mtime_ns
    system = cached_system(model_revision)
except (OSError, ValueError, KeyError, TypeError, ImportError) as error:
    st.error(f"Sign-in is unavailable. Could not load the saved models: {error}")
    st.stop()

identities = system["identities"]
claim = st.selectbox(
    "Account", list(identities), index=None, placeholder="Choose your name",
    format_func=lambda user_id: identities[user_id]["display_name"].title(),
)
if claim is None:
    st.stop()

folder = identities[claim]["display_name"]
name = folder.title()
attempt = f"{claim}_{model_revision}"

st.subheader("1. Face")
st.caption("Center your face close to the camera, then take a photo.")
photo = st.camera_input("Face photo", label_visibility="collapsed", width=420, key=f"photo_{attempt}")
if photo is None:
    st.stop()

face = check_sample("face", verify_face, photo, claim, system, attempt)
if not face["accepted"]:
    st.error(f"Could not verify your face for {name}. Clear the photo and try again.")
    st.stop()
st.success("Face verified.", icon=":material/check_circle:")

face_attempt = f"{attempt}_{photo.file_id}"
st.subheader("2. Voice")
st.caption(f"Choose your WAV recording from `datasets/demo/{folder}`.")
voice = st.file_uploader("Voice recording", type=["wav"], key=f"voice_{face_attempt}")
if voice is None:
    st.stop()

speaker = check_sample("voice", verify_voice, voice, claim, system, face_attempt)
if not speaker["accepted"]:
    message = "Choose a readable WAV recording with a clear, short utterance."
    if speaker["predicted_identity"] is not None:
        message = f"Voice did not verify for {name}. Choose another recording."
    st.error(message)
    st.stop()
st.success("Voice verified.", icon=":material/check_circle:")

voice_attempt = f"{face_attempt}_{voice.file_id}"
st.subheader("3. Fingerprint")
st.caption(f"Choose your fingerprint image from `datasets/demo/{folder}`.")
fingerprint = st.file_uploader(
    "Fingerprint image", type=["png", "jpg", "jpeg", "bmp", "tif", "tiff"],
    key=f"fingerprint_{voice_attempt}",
)
if fingerprint is None:
    st.stop()

finger = check_sample("fingerprint", verify_fingerprint, fingerprint, claim, system, voice_attempt)
if not finger["accepted"]:
    message = "Choose a readable fingerprint image."
    if finger["predicted_identity"] is not None:
        message = f"Fingerprint did not verify for {name}. Choose another image."
    st.error(message)
    st.stop()

st.success(f"Signed in as {name}.", icon=":material/check_circle:")

## Check the Sign-in Steps with Demo Samples

The Streamlit test runner supplies the reference photo at the camera widget and uploads the two demo files. The page uses the saved models for each check. This exercises the sign-in logic with prepared samples.


In [3]:
from streamlit.testing.v1 import AppTest

page = AppTest.from_file(str(ROOT / 'app.py'), default_timeout=60).run()
observed = []
for user_id, identity in system['identities'].items():
    name = identity['display_name']
    folder = ROOT / 'datasets' / 'demo' / name
    page.selectbox[0].select(user_id).run()
    assert not page.file_uploader, 'A photo must be verified before an upload.'
    page.camera_input[0].upload('face.jpg', (folder / 'face.jpg').read_bytes()).run()
    page.file_uploader[0].upload('voice.wav', (folder / 'voice.wav').read_bytes(), 'audio/wav').run()
    page.file_uploader[1].upload('fingerprint.png', (folder / 'fingerprint.png').read_bytes(), 'image/png').run()
    assert not page.exception and not page.error
    assert page.success[-1].value == f'Signed in as {name.title()}.'
    observed.append({'account': name.title(), 'result': page.success[-1].value})
display(pd.DataFrame(observed))


2026-11-11 09:20:24.456 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


,account,result
0,Abdelrahman,Signed in as Abdelrahman.
1,Kareem,Signed in as Kareem.
2,Renad,Signed in as Renad.


## Start the Local Server
**Input:** the selected project interpreter. **Operation:** launch Streamlit in a separate process if START_SERVER is set to True. **Output:** a local URL. If port 8501 is occupied, choose another PORT.

The default is False so Run All does not create duplicate servers. A server may already have been started during project verification. You can also use the exact printed command in a terminal. Stop only the process started by this notebook with the final cell.

In [4]:
PORT = 8501
START_SERVER = False
command = [sys.executable, '-m', 'streamlit', 'run', str(ROOT / 'app.py'),
           '--server.port', str(PORT), '--server.address', '127.0.0.1']
print(subprocess.list2cmdline(command))
print(f'Local URL: http://127.0.0.1:{PORT}')
if START_SERVER:
    if 'server_process' in globals() and server_process.poll() is None:
        print('The server started by this notebook is already running.')
    else:
        server_process = subprocess.Popen(command, cwd=ROOT)
        print('Started server process:', server_process.pid)

"<project directory>\.venv\Scripts\python.exe" -m streamlit run "<project directory>\app.py" --server.port 8501 --server.address 127.0.0.1
Local URL: http://127.0.0.1:8501


## Stop This Notebook's Server

Leave `STOP_SERVER = False` to keep the demo available. The stop cell applies to the process started by this notebook.


In [5]:
STOP_SERVER = False
if STOP_SERVER and 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=15)
    print('Stopped this notebook server.')
else:
    print('No server stopped.')

No server stopped.
